# Iteratively busca Wikipedia com Claude

[DISCLAIMER: este notebook was created using Claude 2 models e is considered legacy.]

alguns questions can't be answered by Claude off the topo of Claude's Cabeça. Maybe they're sobre atual Eventos. Maybe you have an intensely detailed question aquele Claude hasn't memorized the answer para. No worries! com alguns prompting e scaffolding, Claude can buscar the web para encontrar answers. In este notebook, we will criar a virtual research assistant quem has the ability para buscar Wikipedia para encontrar answers para your question. The same approach can be used para allow Claude para buscar the broader web, ou a definir of documents you provide.

o que is the approach? Broadly isso falls under the category of "tool use". We criar a buscar tool, tell Claude sobre isso, e let isso go para work. In pseudocode:

1. Prompt Claude com a Descrição of the buscar tool, como isso's best used, e como para "call" isso (by issuing a special texto).
2. Tell Claude your question.
3. Claude produces tokens like normal. se isso produces the special texto, terminate the token produção stream, e problema a query para a buscar api.
4. Construct a novo prompt qual consists of the prompt de step 1, plus everything Claude generated up para the buscar call texto, plus the results of the api call.
5. Repeat until Claude decides isso's done.

Let's zoom in on the prompts para tool use e retrieval.

### Prompts

In [101]:
# Tool Descrição Prompt
wikipedia_prompt = """You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia para páginas similar to your query. isso Retorna para each page its title and full page content. Use this tool se you want to obter up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. para example, se the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: se the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think palavras-chave, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then obter results back in <search_result> tags.</tool_description>"""
imprimir(wikipedia_prompt)

You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia para páginas similar to your query. isso Retorna para each page its title and full page content. Use this tool se you want to obter up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. para example, se the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: se the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think palavras-chave, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then obter 

Notice aquele ali is a lot of advice in este prompt sobre como para buscar Wikipedia properly. We're todos used para just typing random nonsense into Google e getting decent results because the query parsing logic is so good. Wikipedia buscar is não like aquele. As an example: consider the query "o que's the best way para purchase potatoes in the United Arab Emirates". The [topo hits para este on Wikipedia](https://en.wikipedia.org/w/index.php?buscar=o que%27s+the+best+way+para+purchase+potatoes+in+the+United+Arab+Emirates&title=Special:buscar&perfil=avançado&fulltext=1&ns0=1) are para Slavery in the United States, 1973 Oil Crisis, Wendy's, e Tim Horton's (??). Meanwhile Google correctly takes you straight para Carrefour UAE.

Another difference is aquele Wikipedia buscar retorna entire páginas. com vector buscar, you might be getting narrower chunks, so you might want para ask para mais results, use a mais specific query, ou both. The big-picture takeaway is aquele your results can vary a lot on your choices aqui so pay attention!

In [103]:
retrieval_prompt = """Before beginning to research the user's question, first think para a moment inside <scratchpad> tags about what information is necessary para a well-informed answer. se the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will retornar empty search results, or the search results may not contain the information you need. In such cases, feel free to tentar again with a different query. 

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. se you have all the relevant information, escrever isso in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy."""
imprimir(retrieval_prompt)

Before beginning to research the user's question, first think para a moment inside <scratchpad> tags about what information is necessary para a well-informed answer. se the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will retornar empty search results, or the search results may not contain the information you need. In such cases, feel free to tentar again with a different query. 

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. se you have all the relevant information, escrever isso in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy.


We use a scratchpad aqui para the normal chain-of-thought reasons -- isso makes Claude come up com a coherent plan para answer the question. The buscar quality reflection is used para induce Claude para be persistent e não jump the gun by answering the question antes gathering todos the relevant information. But por que are we telling Claude para synthesize the information e não answer direita away?

In [104]:
answer_prompt = "Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information."
imprimir(answer_prompt)

Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information.


By extracting the information e presenting isso para Claude in a novo query, we allow Claude para focus todos its attention on synthesizing the information into the direita answer. sem este step, we found aquele Claude would às vezes precommit para an answer e então "justify" isso com the buscar results, rather than allowing the results para Guia isso.

agora follows a bunch of code aquele implementa the pseudocode para busca + retrieving + reprompting.

### buscar Implementation

In [88]:
from dataclasses importar dataclass
from abc importar ABC, abstractmethod
importar wikipedia, re
from anthropic importar Anthropic, HUMAN_PROMPT, AI_PROMPT
from typing importar Tuple, Optional

@dataclass
classe SearchResult:
    """
    A single search result.
    """
    content: str

classe SearchTool:
    """
    A search tool that can run a query and retornar a formatted texto of search results.
    """

    def __init__():
        pass

    @abstractmethod
    def raw_search(self, query: str, n_search_results_to_use: int) -> list[SearchResult]:
        """
        Runs a query using the searcher, then Retorna the raw search results without formatting.

        :param query: The query to run.
        :param n_search_results_to_use: The número of results to retornar.
        """
        raise NotImplementedError()
    
    @abstractmethod
    def process_raw_search_results(
        self, results: list[SearchResult],
    ) -> list[str]:
        """
        Extracts the raw search content from the search results and Retorna a list of strings that can be Passou to Claude.

        :param results: The search results to extract.
        """
        raise NotImplementedError()
    
    def search_results_to_string(self, extracted: list[str]) -> str:
        """
        Joins and formats the extracted search results as a texto.

        :param extracted: The extracted search results to formatar.
        """
        result = "\n".juntar(
            [
                f'<item index="{i+1}">\n<page_content>\n{r}\n</page_content>\n</item>'
                para i, r in enumerate(extracted)
            ]
        )
        retornar result

    def wrap_search_results(self, extracted: list[str]) -> str:
        """
        Formats the extracted search results as a texto, including the <search_results> tags.

        :param extracted: The extracted search results to formatar.
        """
        retornar f"\n<search_results>\n{self.search_results_to_string(extracted)}\n</search_results>"
    
    def search(self, query: str, n_search_results_to_use: int) -> str:
        raw_search_results = self.raw_search(query, n_search_results_to_use)
        processed_search_results = self.process_raw_search_results(raw_search_results)
        displayable_search_results = self.wrap_search_results(processed_search_results)
        retornar displayable_search_results 

In [89]:
@dataclass
classe WikipediaSearchResult(SearchResult):
    title: str
    
classe WikipediaSearchTool(SearchTool):

    def __init__(self,
                 truncate_to_n_tokens: Optional[int] = 5000):
        self.truncate_to_n_tokens = truncate_to_n_tokens
        se truncate_to_n_tokens is not None:
            self.tokenizer = Anthropic().get_tokenizer()

    def raw_search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        search_results = self._search(query, n_search_results_to_use)
        retornar search_results
    
    def process_raw_search_results(self, results: list[WikipediaSearchResult]) -> list[str]:
        processed_search_results = [f'Page Title: {result.title.limpar()}\nPage Content:\n{self.truncate_page_content(result.content)}' para result in results]
        retornar processed_search_results

    def truncate_page_content(self, page_content: str) -> str:
        se self.truncate_to_n_tokens is None:
            retornar page_content.limpar()
        senão:
            retornar self.tokenizer.decodificar(self.tokenizer.codificar(page_content).ids[:self.truncate_to_n_tokens]).limpar()
        
    def _search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        results: list[str] = wikipedia.search(query)
        search_results: list[WikipediaSearchResult] = []
        para result in results:
            se len(search_results) >= n_search_results_to_use:
                parar
            tentar:
                page = wikipedia.page(result)
                imprimir(page.url)
            except:
                # The Wikipedia api is a little flaky, so we just skip over páginas that fail to load
                continuar
            content = page.content
            title = page.title
            search_results.anexar(WikipediaSearchResult(content=content, title=title))
        retornar search_results

In [100]:
def extract_between_tags(tag: str, texto: str, limpar: bool = verdadeiro) -> list[str]:
    ext_list = re.findall(f"<{tag}\s?>(.+?)</{tag}\s?>", texto, re.DOTALL)
    se limpar:
        ext_list = [e.limpar() para e in ext_list]
    retornar ext_list

classe ClientWithRetrieval(Anthropic):

    def __init__(self, search_tool: SearchTool, verbose: bool = verdadeiro, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.search_tool = search_tool
        self.verbose = verbose

    # Helper Métodos
    def _search_query_stop(self, partial_completion: str, n_search_results_to_use: int) -> Tuple[list[SearchResult], str]:
        search_query = extract_between_tags('search_query', partial_completion + '</search_query>') 
        se search_query is None:
            raise Exception(f'Completion with retrieval Falhou as partial completion returned mismatched <search_query> tags.')
        imprimir(f'Executando search query against SearchTool: {search_query}')
        search_results = self.search_tool.raw_search(search_query, n_search_results_to_use)
        extracted_search_results = self.search_tool.process_raw_search_results(search_results)
        formatted_search_results = self.search_tool.wrap_search_results(extracted_search_results)
        retornar search_results, formatted_search_results
    
    def retrieve(self,
                       query: str,
                       model: str,
                       n_search_results_to_use: int = 3,
                       stop_sequences: list[str] = [HUMAN_PROMPT],
                       max_tokens_to_sample: int = 1000,
                       max_searches_to_try: int = 5,
                       temperature: float = 1.0) -> tuple[list[SearchResult], str]:
        
        prompt = f"{HUMAN_PROMPT} {wikipedia_prompt} {retrieval_prompt.formatar(query=query)}{AI_PROMPT}"
        starting_prompt = prompt
        imprimir("Starting prompt:", starting_prompt)
        token_budget = max_tokens_to_sample
        all_raw_search_results: list[SearchResult] = []
        para tries in range(max_searches_to_try):
            partial_completion = self.completions.create(prompt = prompt,
                                                     stop_sequences=stop_sequences + ['</search_query>'],
                                                     model=model,
                                                     max_tokens_to_sample = token_budget,
                                                     temperature = temperature)
            partial_completion, stop_reason, stop_seq = partial_completion.completion, partial_completion.stop_reason, partial_completion.stop
            imprimir(partial_completion)
            token_budget -= self.count_tokens(partial_completion)
            prompt += partial_completion
            se stop_reason == 'stop_sequence' and stop_seq == '</search_query>':
                imprimir(f'Attempting search número {tries}.')
                raw_search_results, formatted_search_results = self._search_query_stop(partial_completion, n_search_results_to_use)
                prompt += '</search_query>' + formatted_search_results
                all_raw_search_results += raw_search_results
            senão:
                parar
        final_model_response = prompt[len(starting_prompt):]
        retornar all_raw_search_results, final_model_response
    
    # principal Métodos
    def completion_with_retrieval(self,
                                        query: str,
                                        model: str,
                                        n_search_results_to_use: int = 3,
                                        stop_sequences: list[str] = [HUMAN_PROMPT],
                                        max_tokens_to_sample: int = 1000,
                                        max_searches_to_try: int = 5,
                                        temperature: float = 1.0) -> str:
        
        _, retrieval_response = self.retrieve(query, model=model,
                                                 n_search_results_to_use=n_search_results_to_use, stop_sequences=stop_sequences,
                                                 max_tokens_to_sample=max_tokens_to_sample,
                                                 max_searches_to_try=max_searches_to_try,
                                                 temperature=temperature)
        information = extract_between_tags('information', retrieval_response)[-1]
        prompt = f"{HUMAN_PROMPT} {answer_prompt.formatar(query=query, information=information)}{AI_PROMPT}"
        imprimir("Summarizing:\n", prompt)
        answer = self.completions.create(
            prompt = prompt, model=model, temperature=temperature, max_tokens_to_sample=1000
        ).completion
        retornar answer

### Executando a Query

We're ready para execute a query! let's pick something:
- recente, so isso's menos likely para be in Claude's training data, e
- compound/complexo so isso requires multiple searches.

In [98]:
importar os
# Create a searcher
wikipedia_search_tool = WikipediaSearchTool()
ANTHROPIC_SEARCH_MODEL = "claude-2"

client = ClientWithRetrieval(api_key=os.environ['ANTHROPIC_API_KEY'], verbose=verdadeiro, search_tool = wikipedia_search_tool)

query = "Which movie came out first: Oppenheimer, or Are You There God isso's Me Margaret?"

augmented_response = client.completion_with_retrieval(
    query=query,
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0)
imprimir(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia para páginas similar to your query. isso Retorna para each page its title and full page content. Use this tool se you want to obter up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. para example, se the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: se the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think palavras-chave, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_qu

Cool, Claude was able para make a plan, execute the queries, e synthesize the information into an accurate answer. Nota: sem the extra information extraction step, Claude would às vezes determine the lançamento dates of the movies correctly but então obter the ordering wrong in its final answer. let's do another.

In [99]:
augmented_response = client.completion_with_retrieval(
    query="Who won the 2023 NBA championship? Who was that team's best player in the year 2009?",
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0)
imprimir(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia para páginas similar to your query. isso Retorna para each page its title and full page content. Use this tool se you want to obter up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. para example, se the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: se the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think palavras-chave, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_qu

e ali you have isso! You may notice aquele the buscar tool code is nice e abstrato e can be adapted para use a buscar api of your choice com minor modifications. Just remember para explain para Claude any tips isso needs para use the tool well. You can even give Claude alguns poucos-shot Exemplos of ideal query plans e query structure para melhorar desempenho further.